# 01 - Exploration du dataset (Retailrocket)
Ce notebook charge `events.csv` puis produit des statistiques descriptives et des visualisations rapides.


## 1) Chargement avec PySpark
Le dataset Retailrocket est constitué d'événements utilisateur-produit (view, addtocart, transaction).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, count

spark = SparkSession.builder.appName('ShopAI-01-Exploration').getOrCreate()

events_path = 'data/raw/events.csv'

events = spark.read.csv(events_path, header=True, inferSchema=True)

events.printSchema()
print('Lignes:', events.count())


## 2) Statistiques descriptives
Nombre d'utilisateurs, de produits et d'interactions.


In [ ]:
nb_users = events.select('visitorid').distinct().count()
nb_items = events.select('itemid').distinct().count()
nb_interactions = events.count()

print('Nombre users:', nb_users)
print('Nombre produits:', nb_items)
print('Nombre interactions:', nb_interactions)


## 3) Distribution des types d'événements
view / addtocart / transaction


In [ ]:
dist = events.groupBy('event').agg(count('*').alias('n')).orderBy(col('n').desc())
dist.show()

import matplotlib.pyplot as plt
import pandas as pd

pdf = dist.toPandas()
plt.figure(figsize=(6,4))
plt.bar(pdf['event'], pdf['n'])
plt.title('Distribution des events')
plt.xlabel('event')
plt.ylabel('nombre')
plt.tight_layout()
plt.show()


## 4) Visualisations (heatmap corrélation + distribution interactions)
La corrélation sur un dataset événementiel est limitée, mais on peut illustrer des relations simples entre compteurs.


In [ ]:
from pyspark.sql.functions import when, sum as spark_sum

tmp = (
    events
    .withColumn('is_view', when(col('event')=='view', 1).otherwise(0))
    .withColumn('is_cart', when(col('event')=='addtocart', 1).otherwise(0))
    .withColumn('is_tx', when(col('event')=='transaction', 1).otherwise(0))
    .groupBy('visitorid')
    .agg(
        spark_sum('is_view').alias('views'),
        spark_sum('is_cart').alias('carts'),
        spark_sum('is_tx').alias('transactions'),
    )
)

pdf = tmp.limit(5000).toPandas()

import seaborn as sns

corr = pdf[['views','carts','transactions']].corr()

plt.figure(figsize=(5,4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap corrélation (échantillon)')
plt.tight_layout()
plt.show()


## 5) Top 10 produits les plus vus


In [ ]:
top10 = (
    events.filter(col('event')=='view')
    .groupBy('itemid')
    .agg(count('*').alias('n_views'))
    .orderBy(col('n_views').desc())
    .limit(10)
)

top10.show(truncate=False)

pdf = top10.toPandas()
plt.figure(figsize=(7,4))
plt.bar(pdf['itemid'].astype(str), pdf['n_views'])
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 produits les plus vus')
plt.tight_layout()
plt.show()
